# 07 · Solving with CVXPY

### Recap & why now
Notebook 06 solved the whole route with one `np.linalg.solve`, because every requirement
was an **equality**: hit this waypoint, match at that join.

Real missions are full of requirements that are not. *Stay under 3 m/s. Do not lean past
30°. Stay inside this corridor.* You cannot write "less than" as a row of $Ac = b$, and
the KKT method has no way to express it — it assumes every constraint is tight, and with
inequalities you do not know which are until you have solved.

### Learning objectives
1. Recognise the general **QP form** with both equalities and inequalities.
2. Write our problem in **CVXPY** and verify it against the KKT answer.
3. Add **speed caps** as sampled inequality rows, and see them bind.
4. Read what a solver means by `optimal` and `infeasible`, and what to do about the second.
5. Judge how finely a continuous-time constraint needs sampling.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

In [ ]:
# === The multi-segment solver from Notebook 06 ===========================

def seg_row(N, seg, t, der):
    """A row of the big constraint matrix that touches only segment `seg`."""
    r = np.zeros(N)
    r[seg*NCOEF:(seg+1)*NCOEF] = deriv_row(NCOEF, t, der)
    return r

def build_cost(times, der=4):
    """Block-diagonal Q: one cost_matrix per segment, stacked along the diagonal."""
    Q = np.zeros((len(times)*NCOEF, len(times)*NCOEF))
    for s, T in enumerate(times):
        Q[s*NCOEF:(s+1)*NCOEF, s*NCOEF:(s+1)*NCOEF] = cost_matrix(NCOEF, T, der)
    return Q

def build_constraints(waypoints, times):
    """Waypoints, rest at both ends, and continuity of velocity/acceleration/jerk at each join."""
    m = len(times); N = m*NCOEF
    rows, vals = [], []
    for s in range(m):                             # Every segment starts and ends on its waypoints.
        rows.append(seg_row(N, s, 0.0, 0));      vals.append(waypoints[s])
        rows.append(seg_row(N, s, times[s], 0)); vals.append(waypoints[s+1])
    for der in (1, 2, 3):                          # At rest, in every sense, at both ends.
        rows.append(seg_row(N, 0, 0.0, der));          vals.append(0.0)
        rows.append(seg_row(N, m-1, times[m-1], der)); vals.append(0.0)
    for s in range(m - 1):                         # The two sides of each join must AGREE...
        for der in (1, 2, 3):
            rows.append(seg_row(N, s, times[s], der) - seg_row(N, s+1, 0.0, der))
            vals.append(0.0)                       # ...but we never say WHAT they agree on.
    return np.array(rows), np.array(vals)

def solve_min_snap_1d(waypoints, times, der=4):
    """Equality-constrained QP, solved through the KKT system. One axis."""
    Q = build_cost(times, der)
    A, b = build_constraints(waypoints, times)
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(Q.shape[0]), b]))
    return sol[:Q.shape[0]].reshape(len(times), NCOEF)      # Drop the Lagrange multipliers.

def sample(coeffs, times, t, der=0):
    """Evaluate the piecewise polynomial at global time t."""
    edges = np.concatenate([[0.0], np.cumsum(times)])
    if t <= 0:         return poly_val(coeffs[0], 0.0, der)
    if t >= edges[-1]: return poly_val(coeffs[-1], times[-1], der)
    s = int(np.searchsorted(edges, t, side="right") - 1)
    return poly_val(coeffs[s], t - edges[s], der)

def min_snap_3d(waypoints, times):
    """Solve each axis separately and wrap the result in the ref(t) interface the cascade wants."""
    W = np.asarray(waypoints, float)
    coeffs = [solve_min_snap_1d(W[:, axis], times) for axis in range(3)]
    total = float(np.sum(times))
    def ref(t):
        t = min(max(t, 0.0), total)
        p = np.array([sample(coeffs[a], times, t, 0) for a in range(3)])
        v = np.array([sample(coeffs[a], times, t, 1) for a in range(3)])
        acc = np.array([sample(coeffs[a], times, t, 2) for a in range(3)])
        if t >= total:
            v = np.zeros(3); acc = np.zeros(3)     # Hold position once the trajectory is finished.
        return p, v, acc
    return ref, total, coeffs

ROUTE = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
g = 9.81
print("solver ready — the standing route has %d waypoints and %d segments, %.1f s total" %
      (len(ROUTE), len(DURATIONS), sum(DURATIONS)))

## 1 · The general form

$$\min_c \;\; c^\top Q c \qquad \text{subject to} \qquad A c = b, \quad G c \le h$$

Ours fits exactly: $Q$ is the snap cost, $Ac = b$ holds the waypoints and continuity, and
$Gc \le h$ is the new part. Each inequality is another row of $G$; the solver works out
which ones are active.

The catch is that our constraints are about *continuous time* — "speed under 3 m/s at
every instant" is infinitely many rows. The standard fix is to **sample**: enforce it at
20–30 points per segment and measure the violation between samples rather than assuming
there is none.

In [ ]:
import cvxpy as cp
print("CVXPY", cp.__version__, "| solvers:", [s for s in cp.installed_solvers()
                                              if s in ("CLARABEL", "SCS", "OSQP")])

WPS = [0.0, 2.0, 2.0, 0.0]
TIMES = [2.0, 1.5, 2.5]
M_SEG, N_VAR = len(TIMES), len(TIMES)*NCOEF
Q = build_cost(TIMES)
A, b = build_constraints(WPS, TIMES)

c_var = cp.Variable(N_VAR)
problem = cp.Problem(cp.Minimize(cp.quad_form(c_var, cp.psd_wrap(Q))), [A @ c_var == b])
problem.solve(solver=cp.CLARABEL)

c_kkt = solve_min_snap_1d(WPS, TIMES).reshape(-1)
print("\nCVXPY : status %s, cost %.6f" % (problem.status, problem.value))
print("KKT   :                cost %.6f" % (c_kkt @ Q @ c_kkt))
print("largest coefficient difference: %.2e" % np.abs(c_var.value - c_kkt).max())
print("\nIdentical — same problem, two ways of solving it. Always do this check when you move a")
print("problem into a solver: if the answers disagree, the bug is in the translation.")

## 2 · A speed limit

In continuous time, $|\dot p(t)| \le v_{\max}$ for all $t$. Sampled, it becomes two rows
per sample point, because an absolute value is two one-sided constraints:

$$\text{row}(t, 1)\, c \le v_{\max}, \qquad -\text{row}(t, 1)\, c \le v_{\max}$$

The `deriv_row` machinery from Notebook 02 is doing all the work again. A constraint is
still *(some derivative) at (some time)* — only the `=` has become a `≤`.

In [ ]:
def solve_capped(times, waypoints, v_max=None, p_max=None, n_samples=25, verbose=False):
    """Minimum snap with optional speed and position caps. Returns (coeffs, cost, status)."""
    m = len(times); N = m*NCOEF
    Qq = build_cost(times); Aq, bq = build_constraints(waypoints, times)
    cvar = cp.Variable(N)
    constraints = [Aq @ cvar == bq]                # Everything from Notebook 06.
    for s in range(m):
        for t_ in np.linspace(0, times[s], n_samples):
            if v_max is not None:
                r = seg_row(N, s, t_, 1)
                constraints += [r @ cvar <= v_max, -r @ cvar <= v_max]
            if p_max is not None:
                constraints += [seg_row(N, s, t_, 0) @ cvar <= p_max]
    prob = cp.Problem(cp.Minimize(cp.quad_form(cvar, cp.psd_wrap(Qq))), constraints)
    for solver in (cp.CLARABEL, cp.SCS):           # Near the edge of feasibility, try a second one.
        try:
            prob.solve(solver=solver); break
        except cp.error.SolverError:
            continue
    else:
        return None, None, "solver_failed"
    if prob.status not in ("optimal", "optimal_inaccurate") or cvar.value is None:
        return None, None, prob.status
    return cvar.value.reshape(m, NCOEF), prob.value, prob.status

grid = np.linspace(0, sum(TIMES), 900)
print("  v_max      status        snap cost     actual peak |v|")
for v_max in (None, 2.5, 1.8, 1.5):
    C_v, cost, status = solve_capped(TIMES, WPS, v_max=v_max)
    if C_v is None:
        print("  %-9s %-13s %s" % (v_max, status, "no trajectory exists")); continue
    peak = max(abs(sample(C_v, TIMES, t_, 1)) for t_ in grid)
    print("  %-9s %-13s %10.2f %16.3f" % (str(v_max), status, cost, peak))

print("\nThe 2.5 m/s cap costs nothing — the unconstrained trajectory already peaked below it, so")
print("the constraint is INACTIVE and the solver ignores it. The 1.5 m/s cap bites: the cost")
print("jumps and the peak sits right on the limit, which is what a binding constraint looks like.")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 3.4))
for v_max, colour in [(None, "C0"), (1.8, "C1"), (1.5, "C3")]:
    C_v, _, _ = solve_capped(TIMES, WPS, v_max=v_max)
    label = "no cap" if v_max is None else r"$v_{max}$ = %.1f" % v_max
    a1.plot(grid, [sample(C_v, TIMES, t_, 1) for t_ in grid], color=colour, lw=1.9, label=label)
    a2.plot(grid, [sample(C_v, TIMES, t_, 0) for t_ in grid], color=colour, lw=1.9)
for v in (1.5, -1.5):
    a1.axhline(v, color="C3", ls=":", lw=1.2)
a1.set_xlabel("time [s]"); a1.set_ylabel("velocity [m/s]"); a1.legend(fontsize=8)
a1.set_title("Speed caps flatten the peaks")
edges = np.concatenate([[0.0], np.cumsum(TIMES)])
a2.plot(edges, WPS, "o", color="k", ms=6)
a2.set_xlabel("time [s]"); a2.set_ylabel("position [m]"); a2.set_title("The waypoints are still hit")
plt.tight_layout(); plt.show()

C_v, _, _ = solve_capped(TIMES, WPS, v_max=1.5, n_samples=25)
fine = np.linspace(0, sum(TIMES), 20001)
worst = max(abs(sample(C_v, TIMES, t_, 1)) for t_ in fine)
print("cap 1.500 m/s enforced at 25 points per segment; on a 20001-point grid the true peak is")
print("%.6f m/s — an overshoot of %.1e. That is what sampling costs, and it is measurable." %
      (worst, worst - 1.5))

## 3 · Infeasible is an answer

Ask for something impossible and the solver says so. That is not a failure mode to be
avoided — it is genuinely useful information, and one of the main reasons to state a
problem as a QP rather than tune it by hand.

Our first segment must cover 2 m in 2 s, so its *average* speed is already 1 m/s, and any
smooth profile peaks above its average. The fix is never "loosen the constraint and
hope": it is to change something physical — more time, or a different waypoint.

In [ ]:
print("  tightening the speed limit on the same timing:")
for v_max in (1.5, 1.2, 1.0):
    _, _, status = solve_capped(TIMES, WPS, v_max=v_max)
    print("    v_max = %.1f m/s -> %s" % (v_max, status))

print("\n  why, without needing a solver to tell us:")
for s in range(len(TIMES)):
    dist = abs(WPS[s+1] - WPS[s])
    print("    segment %d: %.1f m in %.1f s -> average speed %.2f m/s" % (s+1, dist, TIMES[s], dist/TIMES[s]))

print("\n  the honest fix — more time on the segments that need it:")
for scale in (1.0, 1.5, 2.0):
    times_slow = [t_*scale for t_ in TIMES]
    _, cost, status = solve_capped(times_slow, WPS, v_max=1.0)
    print("    times x%.1f (total %5.1f s) -> %-12s %s" %
          (scale, sum(times_slow), status, "" if cost is None else "snap cost %.2f" % cost))

def feasible(scale, v_max=1.0):
    _, _, st = solve_capped([t_*scale for t_ in TIMES], WPS, v_max=v_max)
    return st in ("optimal", "optimal_inaccurate")

lo, hi = 1.0, 4.0
for _ in range(18):
    mid = 0.5*(lo + hi)
    if feasible(mid): hi = mid
    else:             lo = mid
print("\n  bisecting: the trajectory becomes feasible at a time scaling of about %.3f," % hi)
print("  i.e. a total duration of %.2f s instead of %.2f s." % (hi*sum(TIMES), sum(TIMES)))
print("  That is the useful response to 'infeasible': ask what would have to change, and let")
print("  the solver tell you how much. Notebook 08 automates the search.")

## 4 · How finely to sample

Sampling density trades enforcement against solve time. Too few points and the trajectory
slips between them; too many and you are paying for rows that say nothing new, since
neighbouring samples on a smooth polynomial are nearly the same constraint.

Somewhere around 20–30 per segment is the usual sweet spot, and the table below is how you
would establish that for your own problem.

In [ ]:
fine = np.linspace(0, sum(TIMES), 20001)
print("  samples/segment   true peak |v|   violation of the 1.5 cap")
for n_s in (3, 8, 25, 100):
    C_s, _, _ = solve_capped(TIMES, WPS, v_max=1.5, n_samples=n_s)
    peak = max(abs(sample(C_s, TIMES, t_, 1)) for t_ in fine)
    print("  %15d %15.4f %19.4f" % (n_s, peak, peak - 1.5))

import time as _time
print("\n  segments   unknowns   solve time")
for m_seg in (3, 6, 12):
    wp = [0.0] + [1.5*(k % 2) for k in range(m_seg)]
    tms = [2.5]*m_seg
    t0 = _time.time(); solve_capped(tms, wp, v_max=3.0); dt_ = _time.time() - t0
    print("  %9d %10d %11.3f s" % (m_seg, m_seg*NCOEF, dt_))
print("\n  Most of that time is CVXPY BUILDING the problem rather than solving it. A flight")
print("  computer calls the underlying solver directly with a pre-built sparse matrix and gets")
print("  milliseconds, which is how onboard re-planning at a few hertz is possible.")

## 🧪 Try it yourself

**E1.** Why can inequality constraints not be handled by the KKT system? What would you
need to know in advance to make it work?

**E2.** Add an **acceleration** cap on top of the speed cap and find the tightest
acceleration limit that still gives an `optimal` status.

In [ ]:
# --- Solution E1 ---
print("E1: the KKT system solves 'A c = b' exactly — it assumes every constraint is TIGHT. An")
print("    inequality may be tight or slack, and which it is depends on the answer you have not")
print("    computed yet. If you somehow knew the ACTIVE SET in advance you could treat those as")
print("    equalities, drop the rest, and use KKT — which is literally what active-set solvers do,")
print("    guessing the set and correcting. Interior-point solvers like CLARABEL take a different")
print("    route, but the difficulty they are both handling is the same one.")

# --- Solution E2 ---
def solve_with_accel(times, waypoints, v_max, a_max, n_samples=25):
    m = len(times); N = m*NCOEF
    Qq = build_cost(times); Aq, bq = build_constraints(waypoints, times)
    cvar = cp.Variable(N); cons = [Aq @ cvar == bq]
    for s in range(m):
        for t_ in np.linspace(0, times[s], n_samples):
            rv, ra = seg_row(N, s, t_, 1), seg_row(N, s, t_, 2)
            cons += [rv @ cvar <= v_max, -rv @ cvar <= v_max,
                     ra @ cvar <= a_max, -ra @ cvar <= a_max]
    pr = cp.Problem(cp.Minimize(cp.quad_form(cvar, cp.psd_wrap(Qq))), cons)
    try:
        pr.solve(solver=cp.CLARABEL)
    except cp.error.SolverError:
        return "solver_failed", None
    return pr.status, pr.value

print("\nE2: speed capped at 1.8 m/s, tightening the acceleration cap:")
for a_max in (5.0, 3.0, 2.0, 1.5, 1.0):
    status, cost = solve_with_accel(TIMES, WPS, 1.8, a_max)
    print("    a_max = %.1f m/s^2 -> %-22s %s" % (a_max, status, "" if cost is None else "cost %.2f" % cost))
print("    Worth connecting back to the vehicle: Project 5 limited tilt to 35°, and a 35° tilt")
print("    buys g*tan(35°) = %.2f m/s^2 of horizontal acceleration. An acceleration cap in the" % (g*np.tan(np.deg2rad(35))))
print("    PLANNER is how you guarantee the CONTROLLER never has to ask for more tilt than it is")
print("    allowed — the two limits are the same limit, expressed one layer apart.")

## 🚁 Mini-project: watching a constraint bind

Sweep the speed cap from loose to tight and animate the velocity profile flattening
against it. The moment the cap starts touching the curve is the moment the constraint
becomes active — and the snap cost starts climbing.

In [ ]:
caps = np.concatenate([np.linspace(2.4, 1.45, 30), np.linspace(1.45, 2.4, 30)])
solutions = []
for v_max in caps:
    C_v, cost, status = solve_capped(TIMES, WPS, v_max=float(v_max), n_samples=20)
    solutions.append((C_v, cost) if C_v is not None else (None, None))

grid = np.linspace(0, sum(TIMES), 300)
fig, ax = plt.subplots(figsize=(7.4, 3.6))

def frame(k):
    ax.clear()
    C_v, cost = solutions[k]
    if C_v is not None:
        ax.plot(grid, [sample(C_v, TIMES, t_, 1) for t_ in grid], color="C0", lw=2.2)
    ax.axhline(caps[k], color="C3", ls="--", lw=1.5)
    ax.axhline(-caps[k], color="C3", ls="--", lw=1.5)
    ax.set_ylim(-3, 3); ax.set_xlabel("time [s]"); ax.set_ylabel("velocity [m/s]")
    ax.set_title("cap %.2f m/s    snap cost %s" %
                 (caps[k], "infeasible" if cost is None else "%.1f" % cost), fontsize=10)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(solutions), interval=70, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Stating a planner as a convex program is why this part of
> robotics feels solid compared with, say, gain tuning. You say what you want; the solver
> returns the unique best answer or tells you honestly that none exists; and there is no
> hidden local minimum. That property is why QPs also appear in model-predictive control,
> in legged-robot whole-body control, and in rocket landing guidance.

**Where next.** Two loose ends, and they are the same one. The speed cap was infeasible
because the segment times were too short, and Notebook 06's excursion happened because one
segment's time was too long. Nobody has yet said where those times come from.